In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col, current_timestamp

# Define environment volume paths
base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"
batch_landing_path = f"{base_volume}/batch_ingest_landing"
bronze_table = "training_medallion.01_bronze.bronze_tickets_batch"

# Prepare landing directory and sample historical batch dataset
dbutils.fs.mkdirs(batch_landing_path)

historical_payload = """{"ticket_id": "HIST-001", "customer_id": 501, "issue": "Billing Inquiry", "status": "CLOSED"}
{"ticket_id": "HIST-002", "customer_id": 502, "issue": "Password Reset", "status": "CLOSED"}"""

dbutils.fs.put(f"{batch_landing_path}/historical_batch.json", historical_payload, overwrite=True)

# 1. Read static JSON batch data
df_historical = (
    spark.read
    .format("json")
    .load(batch_landing_path)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path")) # Unity Catalog metadata column
)

# 2. Write batch dataset into Bronze Delta Table (Append Mode)
(
    df_historical.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print(f"✅ Successfully completed historical batch load into {bronze_table}")

Wrote 186 bytes.
✅ Successfully completed historical batch load into training_medallion.01_bronze.bronze_tickets_batch


In [0]:
from pyspark.sql.functions import col, current_timestamp

# Define paths for streaming ingestion
stream_landing_path = f"{base_volume}/stream_ingest_landing"
checkpoint_path = f"{base_volume}/_checkpoints/stream_bronze_tickets"
schema_path = f"{base_volume}/_schemas/stream_bronze_tickets"
streaming_bronze_table = "training_medallion.01_bronze.bronze_tickets_stream"

# Prepare streaming landing directory and landing payload
dbutils.fs.mkdirs(stream_landing_path)

stream_payload_1 = """{"ticket_id": "STREAM-101", "customer_id": 601, "issue": "Latency Issue", "status": "OPEN"}
{"ticket_id": "STREAM-102", "customer_id": 602, "issue": "Payment Error", "status": "PENDING"}"""

dbutils.fs.put(f"{stream_landing_path}/stream_batch_1.json", stream_payload_1, overwrite=True)

# 1. Define Auto Loader streaming reader
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(stream_landing_path)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

# 2. Write stream to Bronze Delta Table using availableNow trigger (Serverless Compatible)
streaming_query = (
    df_stream.writeStream
    .format("delta")
    .trigger(availableNow=True) # Runs micro-batch processing then safely stops
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .table(streaming_bronze_table)
)

streaming_query.awaitTermination()
print(f"✅ Successfully ingested streaming batch into {streaming_bronze_table}")

Wrote 186 bytes.
✅ Successfully ingested streaming batch into training_medallion.01_bronze.bronze_tickets_stream


In [0]:
%sql
-- 1. Inspect Batch Bronze Table
SELECT 
  ticket_id, 
  customer_id, 
  issue, 
  status, 
  ingestion_time, 
  source_file 
FROM training_medallion.01_bronze.bronze_tickets_batch;

-- 2. Inspect Streaming Bronze Table
SELECT 
  ticket_id, 
  customer_id, 
  issue, 
  status, 
  ingestion_time, 
  source_file 
FROM training_medallion.01_bronze.bronze_tickets_stream;

-- 3. Check Delta History to audit write operations
DESCRIBE HISTORY training_medallion.01_bronze.bronze_tickets_stream;
DESCRIBE HISTORY training_medallion.01_bronze.bronze_tickets_batch;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-29T14:53:44.000Z,2067248053951322,rifqiacim@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""d58d3025-206e-457d-9608-aa82b9d4f38a"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-7ff18d7d-58ad-4605-ac0e-363f94004251"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-522f12a5-3e1f-424d-a38e-bcb234a1bc27""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(2518055480986581),4c1d45dd-0d35-4c00-b898-f2dfd4e17cb9,1029-141544-rf3qgsl9-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2594)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
